# Tutorial 6: VIO and Trajectory

## Introduction

In Aria-Gen2 glasses, one of the key upgrade from Aria-Gen1 is the capability to run Machine Perception (MP) algorithms on the device during streaming / recording. Currently supported on-device MP algorithms include Eye-tracking, Hand-tracking, and VIO. These algorithm results are stored as separate data streams in the VRS file. 

**VIO (Visual Inertial Odometry)** combines camera images and IMU (Inertial Measurement Unit) data to estimate device pose and motion in real-time. VIO tracks the device's position, orientation, and velocity by performing visual tracking, IMU integration, sensor fusion, etc, making it the foundation for spatial tracking and understanding. 

In Aria-Gen2 devices, the VIO algorithm are run on device to produce 2 types of tracking results as part of the VRS file: VIO and VIO High Frequency.

The same question -- *where was the device* -- is also answered offline, at higher
quality, by **MPS SLAM**: a closed-loop trajectory, a semi-dense point cloud, and
online calibration, all produced in the cloud after upload. This tutorial covers both
sources, because choosing between them is the first decision you make when you need
device pose.

| | On-device VIO | MPS SLAM |
| :-- | :-- | :-- |
| Where | On the glasses, during recording | Cloud, after upload |
| Available | Immediately, and while streaming | After processing completes |
| Drift | Accumulates; odometry frame only | Corrected by loop closure; world frame |
| Extras | -- | Semi-dense point cloud, online calibration |
| Read with | `VrsDataProvider` stream accessors | `mps.read_*` / `MpsDataProvider` |

**What you'll learn:**

- How to access on-device VIO and VIO_high_frequency data from VRS files
- How to visualize 3D trajectory from on-device VIO data
- How to load the MPS closed-loop and open-loop trajectories, and how they differ
- How to load and filter the MPS semi-dense point cloud and its per-frame observations
- How to visualize an MPS trajectory, point cloud and observations together in 3D

**Prerequisites**
- Complete Tutorial 1 (VrsDataProvider Basics) to understand basic data provider concepts
- Complete Tutorial 2 (Device Calibration) to understand how to properly use calibration in Aria data.
- Download Aria Gen2 sample data: [VRS](https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1.vrs) and [MPS output zip file](https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1_mps_output_dec_2025.zip)
- Complete Tutorial 5 (MPS Basics) for how MPS output is laid out and loaded


### ⚠️ Important Notes
- **Google Colab Users:**  
  If you encounter a `ModuleNotFoundError: No module named 'rerun'` error after installing `rerun-sdk`, Colab may not recognize the new package until the runtime is restarted.  
  **Fix:** Go to **Runtime → Restart session and run all**.

- **Visualization Issue :**  
  If a Rerun visualization window does not appear, this may be due to a known caching issue. Simply re-run the visualization cell to resolve it.

## Setup Environment (Google Colab)

If running on Google Colab, install projectaria-tools and download sample data.

In [ ]:
import sys
import os
import subprocess

google_colab_env = 'google.colab' in str(get_ipython())

if google_colab_env:
    print("Running from Google Colab, installing projectaria_tools and downloading sample data")

    # Install projectaria-tools
    !pip install projectaria-tools==2.3.0

    # Set up data path
    vrs_sample_path = "./vrs_sample_data"

    # Sample VRS file and MPS output URLs
    vrs_url = "https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1.vrs"
    mps_url = "https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1_mps_output_dec_2025.zip"

    vrs_filename = "aria_gen2_sample_data_1.vrs"
    mps_zip_filename = "aria_gen2_sample_data_1_mps_output_dec_2025.zip"

    vrs_file_path = os.path.join(vrs_sample_path, vrs_filename)
    mps_zip_path = os.path.join(vrs_sample_path, mps_zip_filename)
    mps_folder_path = os.path.join(vrs_sample_path, "mps_output")

    # Download and unzip commands
    command_list = [
        f"mkdir -p {vrs_sample_path}",
        f'curl -o {vrs_file_path} -C - -O -L "{vrs_url}"',
        f'curl -o {mps_zip_path} -C - -O -L "{mps_url}"',
        f"unzip -o {mps_zip_path} -d {mps_folder_path}"
    ]

    # Execute the commands for downloading dataset
    print(f"Downloading VRS and MPS sample data...")
    for command in command_list:
        !$command

    print(f"Download complete! VRS file saved to: {vrs_file_path}")
    print(f"MPS data extracted to: {mps_folder_path}")

    # Running this command to trigger early failure of importing ReRun.
    # Should be resolved by restarting the Colab session.
    import rerun as rr
else:
    # For local environment, user needs to specify their own paths
    vrs_file_path = "path/to/your/recording.vrs"
    mps_folder_path = "path/to/your/mps/folder/"
    print(f"Please update vrs_file_path and mps_folder_path to point to your data")


In [ ]:
from projectaria_tools.core import data_provider

# Load VRS file
vrs_data_provider = data_provider.create_vrs_data_provider(vrs_file_path)

# Query VIO data streams
vio_label = "vio"
vio_stream_id = vrs_data_provider.get_stream_id_from_label(vio_label)
if vio_stream_id is None:
    raise RuntimeError(
        f"{vio_label} data stream does not exist! Please use a VRS that contains valid VIO data for this tutorial."
    )

# Query VIO_high_frequency data streams
vio_high_freq_label = "vio_high_frequency"
vio_high_freq_stream_id = vrs_data_provider.get_stream_id_from_label(vio_high_freq_label)
if vio_high_freq_stream_id is None:
    raise RuntimeError(
        f"{vio_high_freq_label} data stream does not exist! Please use a VRS that contains valid VIO high frequency data for this tutorial."
    )


## On-Device VIO Data Stream
### Data Type: `FrontendOutput`
This a new data type introduced to store the results from the VIO system, containing the following fields: 

| Field Name                    | Description                                |
| ----------------------------- | ------------------------------------------ |
| `frontend_session_uid`        | Session identifier (resets on VIO restart) |
| `frame_id`                    | Frame set identifier                       |
| `capture_timestamp_ns`        | Center capture time in nanoseconds         |
| `unix_timestamp_ns`           | Unix timestamp in nanoseconds              |
| `status`                      | VIO status (VALID/INVALID)                 |
| `pose_quality`                | Pose quality (GOOD/BAD/UNKNOWN)            |
| `visual_tracking_quality`     | Visual-only tracking quality               |
| `online_calib`                | Online calibration estimates for SLAM cameras and IMUs  |
| `gravity_in_odometry`         | Gravity vector in odometry frame           |
| `transform_odometry_bodyimu`  | Body IMU's pose in odometry reference frame         |
| `transform_bodyimu_device`    | Transform from body IMU to device frame    |
| `linear_velocity_in_odometry` | Linear velocity in odometry frame in m/s         |
| `angular_velocity_in_bodyimu` | Angular velocity in body IMU frame in rad/s      |

Here, **body IMU** is the IMU that is picked as the reference for motion tracking. For Aria-Gen2' on-device VIO algorithm, this is often `imu-left`. 

**Important Note**: Always check `status == VioStatus.VALID` and
`pose_quality == TrackingQuality.GOOD` for VIO data validity!

### Data Access API



In [ ]:
from projectaria_tools.core.sensor_data import VioStatus, TrackingQuality

print("=== VIO Data Sample ===")

# Find the first valid VIO data sample
num_vio_samples = vrs_data_provider.get_num_data(vio_stream_id)
first_valid_index = None
for idx in range(num_vio_samples):
    vio_data = vrs_data_provider.get_vio_data_by_index(vio_stream_id, idx)
    if (
        vio_data.status == VioStatus.VALID
        and vio_data.pose_quality == TrackingQuality.GOOD
    ):
        first_valid_index = idx
        break

if first_valid_index is not None:
    print("=" * 50)
    print(f"First VALID VIO Data Sample (Index: {first_valid_index})")
    print("=" * 50)

    # Session Information
    print(f"Session UID: {vio_data.frontend_session_uid}")
    print(f"Frame ID: {vio_data.frame_id}")

    # Timestamps
    print(f"Capture Time: {vio_data.capture_timestamp_ns} ns")
    print(f"Unix Time: {vio_data.unix_timestamp_ns} ns")

    # Quality Status
    print(f"Status: {vio_data.status}")
    print(f"Pose Quality: {vio_data.pose_quality}")
    print(f"Visual Quality: {vio_data.visual_tracking_quality}")

    # Transforms
    print(f"Transform Odometry → Body IMU:\n{vio_data.transform_odometry_bodyimu.to_matrix()}")
    print(f"Transform Body IMU → Device:\n{vio_data.transform_bodyimu_device.to_matrix()}")

    # Motion
    print(f"Linear Velocity: {vio_data.linear_velocity_in_odometry}")
    print(f"Angular Velocity: {vio_data.angular_velocity_in_bodyimu}")
    print(f"Gravity Vector: {vio_data.gravity_in_odometry}")
else:
    print("⚠️  No valid VIO sample found")


## On-Device VIO High Frequency Data Stream

**VIO High Frequency** results are generated directly from the on-device VIO results by performing IMU integration between VIO poses, hence provides a much higher data rate at approximately **800Hz**. 

### Data Type: `OpenLoopTrajectoryPose`
The **VioHighFrequency** stream **re-uses** the `OpenLoopTrajectoryPose` data
structure [defined in MPS](https://github.com/facebookresearch/projectaria_tools/blob/main/core/mps/Trajectory.h). 

| Field Name                        | Description                                             |
| --------------------------------- | ------------------------------------------------------- |
| `tracking_timestamp`              | Timestamp in device time domain, in microseconds        |
| `transform_odometry_device`       | Transformation from device to odometry coordinate frame, represented as a SE3 instance. |
| `device_linear_velocity_odometry` | Translational velocity of device in odometry frame, in m/s     |
| `angular_velocity_device`         | Angular velocity of device in device frame, in rad/s              |
| `quality_score`                   | Quality of pose estimation (higher = better)            |
| `gravity_odometry`                | Earth gravity vector in odometry frame                  |
| `session_uid`                     | Unique identifier for VIO tracking session              |

**Important Note**: Due to the high frequency nature of this data (~800Hz), consider
subsampling for visualization to maintain performance.

In [ ]:
print("=== VIO High-Frequency Data Sample ===")

# Find the first VIO high_frequency data sample with high quality value
num_vio_high_freq_samples = vrs_data_provider.get_num_data(vio_high_freq_stream_id)
first_valid_index = None
for idx in range(num_vio_samples):
    vio_high_freq_data = vrs_data_provider.get_vio_high_freq_data_by_index(vio_high_freq_stream_id, idx)
    if (
        vio_high_freq_data.quality_score > 0.5
    ):
        first_valid_index = idx
        break

if first_valid_index is not None:
    print("=" * 50)
    print(f"First VIO High Freq Data Sample with good quality score (Index: {first_valid_index})")
    print("=" * 50)

    # Timestamps, convert timedelta to nanoseconds
    capture_timestamp_ns = int(vio_high_freq_data.tracking_timestamp.total_seconds() * 1e9)

    # Session Information
    print(f"Session UID: {vio_high_freq_data.session_uid}")

    # Timestamps
    print(f"Tracking Time: {capture_timestamp_ns} ns")

    # Quality
    print(f"Quality Score: {vio_high_freq_data.quality_score:.3f}")

    # Transform
    print(f"Transform Odometry → Device:\n{vio_high_freq_data.transform_odometry_device.to_matrix()}")

    # Motion
    print(f"Linear Velocity: {vio_high_freq_data.device_linear_velocity_odometry}")
    print(f"Angular Velocity: {vio_high_freq_data.angular_velocity_device}")
    print(f"Gravity Vector: {vio_high_freq_data.gravity_odometry}")


## Visualizing On-Device VIO trajectory

The following code snippets demonstrate how to visualize a VIO trajectory, along with the
glasses outline and the gravity direction, in a 3D view. 

In [ ]:
import rerun as rr
import numpy as np
from projectaria_tools.core.sensor_data import SensorDataType, TimeDomain, TimeQueryOptions
from projectaria_tools.utils.rerun_helpers import AriaGlassesOutline, ToTransform3D

print("\n=== Visualizing on-device VIO trajectory in 3D view ===")

rr.init("rerun_viz_vio_trajectory")

device_calib = vrs_data_provider.get_device_calibration()

# Set up a data queue
deliver_options = vrs_data_provider.get_default_deliver_queued_options()
deliver_options.deactivate_stream_all()
deliver_options.activate_stream(vio_stream_id)

# Play for only 3 seconds
total_length_ns = vrs_data_provider.get_last_time_ns_all_streams(TimeDomain.DEVICE_TIME) - vrs_data_provider.get_first_time_ns_all_streams(TimeDomain.DEVICE_TIME)
skip_begin_ns = int(15 * 1e9) # Skip 15 seconds
duration_ns = int(3 * 1e9) # 3 seconds
skip_end_ns = max(total_length_ns - skip_begin_ns - duration_ns, 0)
deliver_options.set_truncate_first_device_time_ns(skip_begin_ns)
deliver_options.set_truncate_last_device_time_ns(skip_end_ns)

# Plot VIO trajectory in 3D view.
# Need to keep a cache to store already-loaded trajectory
vio_traj_cached_full = []
for sensor_data in vrs_data_provider.deliver_queued_sensor_data(deliver_options):
    # Convert sensor data to VIO data
    vio_data = sensor_data.vio_data()

    # Check VIO data validity, only plot for valid data
    if ( vio_data.status != VioStatus.VALID or vio_data.pose_quality != TrackingQuality.GOOD):
        print(f"VIO data is invalid for timestamp {sensor_data.get_time_ns(TimeDomain.DEVICE_TIME)}")
        continue

    # Set timestamp
    rr.set_time("device_time", duration=np.timedelta64(vio_data.capture_timestamp_ns, "ns"))

    # Set and plot the Device pose for the current timestamp, as a RGB axis
    T_World_Device = (
        vio_data.transform_odometry_bodyimu @ vio_data.transform_bodyimu_device
    )
    rr.log(
        "world/device",
        ToTransform3D(T_World_Device),
    )
    rr.log(
        "world/device",
        rr.TransformAxes3D(axis_length=0.05),
    )

    # Also plot Aria glass outline for visualization
    aria_glasses_point_outline = AriaGlassesOutline(
        device_calib, use_cad_calib=True
    )
    rr.log(
        "world/device/glasses_outline",
        rr.LineStrips3D(
            aria_glasses_point_outline,
            colors=[200,200,200],
            radii=5e-4,
        ),
    )

    # Plot gravity direction vector
    rr.log(
        "world/vio_gravity",
        rr.Arrows3D(
            origins=[T_World_Device.translation()[0]],
            vectors=[
                vio_data.gravity_in_odometry * 1e-2
            ],  # length converted from 9.8 meter -> 10 cm
            colors=[101,67,33],
            radii=1.5e-3,
        ),
        static=False,
    )

    # Plot VIO trajectory that are cached so far
    vio_traj_cached_full.append(T_World_Device.translation()[0])
    rr.log(
        "world/vio_trajectory",
        rr.LineStrips3D(
            vio_traj_cached_full,
            colors=[173, 216, 255],
            radii=1.5e-3,
        ),
        static=False,
    )

rr.notebook_show()

---

# MPS trajectory and point cloud

Everything above came off the glasses in real time. The rest of this tutorial covers
the same quantity computed in the cloud, which is what you use when accuracy matters
more than latency.

MPS output is a folder of CSV files, not a VRS stream, so it is loaded with the
`mps` module rather than through `vrs_data_provider`. `Tutorial_5_mps_basics` covers
the folder layout and the two loading styles; this tutorial goes straight to the SLAM
outputs.

## Output files

MPS output result files are categorized into sub-folders by algorithms. 
For SLAM algorithm output, it generates the following files: 
- `closed_loop_trajectory.csv`
- `open_loop_trajectory.csv`
- `semidense_observations.csv.gz`
- `semidense_points.csv.gz`
- `online_calibration.jsonl`
- `summary.json`

Please refer to the [MPS Wiki page](https://facebookresearch.github.io/projectaria_tools/docs/data_formats/mps/slam) for details of each file. 

## Closed vs open loop trajectory

MPS SLAM algorithm outputs 2 trajectory files (see [wiki page](https://facebookresearch.github.io/projectaria_tools/docs/data_formats/mps/slam/mps_trajectory) for data type definitions): 
- **Open loop trajectory**: High-frequency (1kHz) odometry from visual-inertial odometry (VIO), accurate over short periods but drifts over time and distance.
- **Closed loop trajectory**: High-frequency (1kHz) pose from mapping with loop closure corrections, reducing drift but possibly less accurate locally over short spans.

In [ ]:
import os

from projectaria_tools.core import mps
from projectaria_tools.core.mps.utils import (
    filter_points_from_confidence,
    get_nearest_pose,
)

print("=== MPS - Closed loop trajectory ===")

# Load MPS closed-loop trajectory data
closed_loop_trajectory_file = os.path.join(
    mps_folder_path, "slam", "closed_loop_trajectory.csv"
)
closed_loop_trajectory = mps.read_closed_loop_trajectory(closed_loop_trajectory_file)

# Print out the content of the first sample in closed_loop_trajectory
if closed_loop_trajectory:
    sample = closed_loop_trajectory[0]
    print("ClosedLoopTrajectoryPose sample:")
    print(f"  tracking_timestamp: {int(sample.tracking_timestamp.total_seconds() * 1e6)} us")
    print(f"  utc_timestamp: {int(sample.utc_timestamp.total_seconds() * 1e6)} us")
    print(f"  transform_world_device:\n{sample.transform_world_device}")
    print(f"  device_linear_velocity_device: {sample.device_linear_velocity_device}")
    print(f"  angular_velocity_device: {sample.angular_velocity_device}")
    print(f"  quality_score: {sample.quality_score}")
    print(f"  gravity_world: {sample.gravity_world}")
    print(f"  graph_uid: {sample.graph_uid}")
else:
    print("closed_loop_trajectory is empty.")


print("=== MPS - Open loop trajectory ===")

# Load MPS open-loop trajectory data
open_loop_trajectory_file = os.path.join(
    mps_folder_path, "slam", "open_loop_trajectory.csv"
)
open_loop_trajectory = mps.read_open_loop_trajectory(open_loop_trajectory_file)

# Print out the content of the first sample in open_loop_trajectory
if open_loop_trajectory:
    sample = open_loop_trajectory[0]
    print("OpenLoopTrajectoryPose sample:")
    print(f"  tracking_timestamp: {int(sample.tracking_timestamp.total_seconds() * 1e6)} us")
    print(f"  utc_timestamp: {int(sample.utc_timestamp.total_seconds() * 1e6)} us")
    print(f"  transform_odometry_device:\n{sample.transform_odometry_device}")
    print(f"  device_linear_velocity_odometry: {sample.device_linear_velocity_odometry}")
    print(f"  angular_velocity_device: {sample.angular_velocity_device}")
    print(f"  quality_score: {sample.quality_score}")
    print(f"  gravity_odometry: {sample.gravity_odometry}")
    print(f"  session_uid: {sample.session_uid}")
else:
    print("open_loop_trajectory is empty.")

## Semi-dense point cloud and observations

MPS SLAM algorithm outputs 2 files related to semi-dense point cloud (see [wiki page](https://facebookresearch.github.io/projectaria_tools/docs/data_formats/mps/slam/mps_pointcloud) for data type definitions): 
- `semidense_points.csv.gz`: Global points in the world coordinate frame. 
- `semidense_observations.csv.gz`: Point observations for each camera, at each timestamp.

Note that semidense point files are normally large, therefore loading them may take some time. 

In [ ]:
print("=== MPS - Semi-dense Point Cloud ===")

# Load MPS semi-dense point cloud data
semidense_points_file = os.path.join(
    mps_folder_path, "slam", "semidense_points.csv.gz"
)
semidense_points = mps.read_global_point_cloud(semidense_points_file)

# Print out the content of the first sample in semidense_points
if semidense_points:
    sample = semidense_points[0]
    print("GlobalPointPosition sample:")
    print(f"  uid: {sample.uid}")
    print(f"  graph_uid: {sample.graph_uid}")
    print(f"  position_world: {sample.position_world}")
    print(f"  inverse_distance_std: {sample.inverse_distance_std}")
    print(f"  distance_std: {sample.distance_std}")
    print(f"Total number of semi-dense points: {len(semidense_points)}")
else:
    print("semidense_points is empty.")

# Filter semidense points by inv_dep or depth.
# The filter will KEEP points with (inv_dep or depth < threshold)
filtered_semidense_points = filter_points_from_confidence(raw_points = semidense_points, threshold_invdep = 1e-3, threshold_dep = 5e-2)
print(f"Filtering semidense points from a total of {len(semidense_points)} points down to {len(filtered_semidense_points)}")

In [ ]:
print("=== MPS - Semi-dense Point Observations ===")

# Load MPS semi-dense point observations data
semidense_observations_file = os.path.join(
    mps_folder_path, "slam", "semidense_observations.csv.gz"
)
semidense_observations = mps.read_point_observations(semidense_observations_file)

# Print out the content of the first sample in semidense_observations
if semidense_observations:
    sample = semidense_observations[0]
    print("PointObservation sample:")
    print(f"  point_uid: {sample.point_uid}")
    print(f"  frame_capture_timestamp: {int(sample.frame_capture_timestamp.total_seconds() * 1e6)} us")
    print(f"  camera_serial: {sample.camera_serial}")
    print(f"  uv: {sample.uv}")
    print(f"Total number of point observations: {len(semidense_observations)}")
else:
    print("semidense_observations is empty.")

## Visualizing the MPS trajectory in 3D

The cells below prepare a short trajectory segment, attach the semi-dense points that
were observed from `slam-front-left` at each of its timestamps, and colour those points
by their depth from the device. Everything is then logged to Rerun as one 3D scene.

In [ ]:
from collections import defaultdict

import numpy as np

# A helper coloring function
def color_from_zdepth(z_depth_m: float) -> np.ndarray:
    """
    Map z-depth (meters, along the camera's forward axis) to a bright Viridis-like RGB color.
    - If z_depth_m <= 0 (point is behind the camera), return white [255, 255, 255].
    - Near (0.2 m) -> yellow, Far (3.0 m) -> purple.
    Returns an array of shape (3,) with dtype=uint8.
    """
    if not np.isfinite(z_depth_m) or z_depth_m <= 0.0:
        return np.array([0, 0, 0], dtype=np.uint8)

    NEAR_METERS, FAR_METERS = 0.2, 5.0

    # Normalize to [0,1], then flip so near → bright (yellow), far → dark (purple)
    clamped = min(max(float(z_depth_m), NEAR_METERS), FAR_METERS)
    normalized_position = (clamped - NEAR_METERS) / (FAR_METERS - NEAR_METERS + 1e-12)
    gradient_position = 1.0 - normalized_position

    # Viridis-like anchor colors: purple → blue → teal → green → yellow
    color_stops = [
        (68, 1, 84),
        (59, 82, 139),
        (33, 145, 140),
        (94, 201, 98),
        (253, 231, 37),
    ]

    # Locate segment and blend between its endpoints
    segment_count = len(color_stops) - 1
    continuous_index = gradient_position * segment_count
    lower_segment_index = int(continuous_index)

    if lower_segment_index >= segment_count:
        red, green, blue = color_stops[-1]
    else:
        segment_fraction = continuous_index - lower_segment_index
        r0, g0, b0 = color_stops[lower_segment_index]
        r1, g1, b1 = color_stops[lower_segment_index + 1]
        red   = r0 + segment_fraction * (r1 - r0)
        green = g0 + segment_fraction * (g1 - g0)
        blue  = b0 + segment_fraction * (b1 - b0)

    return np.array([int(red), int(green), int(blue)], dtype=np.uint8)

In [ ]:
print("=== Preparing MPS SLAM results for visualization ===")

# Check if we have valid SLAM data to visualize
if not closed_loop_trajectory or not semidense_points:
    raise RuntimeError("Warning: This tutorial requires valid MPS SLAM data to run.")

# -----------
# Prepare Trajectory data
# -----------
# Select a short segment of trajectory (e.g., first 5000 samples, subsampled by 50)
segment_length = min(50000, len(closed_loop_trajectory))
trajectory_segment = closed_loop_trajectory[:segment_length:50]
timestamp_to_pose = {
    pose.tracking_timestamp: pose for pose in trajectory_segment
}
print(f"Finished preparing a trajectory of length {len(trajectory_segment)}... ")

# -----------
# Prepare Semidense point data
# -----------
# Filter the semidense point cloud by confidence and limit max point count, and extract the point positions
filtered_semidense_point_cloud_data = filter_points_from_confidence(semidense_points)
points_positions = np.array(
    [
        point.position_world for point in filtered_semidense_point_cloud_data
    ]
)
print(f"Finished preparing filtered semidense points cloud of {len(filtered_semidense_point_cloud_data)} points... ")

# -----------
# Prepare Semidense observation data
# -----------
# Based on RGB observations, create a per-timestamp point position list, and color them according to its distance from RGB camera
point_uid_to_position = {
    point.uid: np.array(point.position_world) for point in filtered_semidense_point_cloud_data
}

# A helper function that creates a easier-to-query mapping to obtain observations according to timestamps
slam_1_serial = vrs_data_provider.get_device_calibration().get_camera_calib("slam-front-left").get_serial_number()
timestamp_to_point_positions = defaultdict(list)  # t_ns -> [position, position, ...]
timestamp_to_point_colors = defaultdict(list) # t_ns -> [color, color, ...]

for obs in semidense_observations:
    # Only add observations for SLAM_1 camera, and if the timestamp is in the chosen trajectory segment
    if (
        obs.camera_serial == slam_1_serial and
        obs.frame_capture_timestamp in timestamp_to_pose and
        obs.point_uid in point_uid_to_position):
        # Insert point position
        obs_timestamp = obs.frame_capture_timestamp
        point_position = point_uid_to_position[obs.point_uid]
        timestamp_to_point_positions[obs_timestamp].append(point_position)

        # Insert point color
        T_world_device = timestamp_to_pose[obs_timestamp].transform_world_device
        point_in_device = T_world_device.inverse() @ point_position
        point_z_depth = point_in_device.squeeze()[2]
        point_color = color_from_zdepth(point_z_depth)
        timestamp_to_point_colors[obs_timestamp].append(point_color)

from itertools import islice
print("Finished preparing semidense points observations: ")
for timestamp, points in islice(timestamp_to_point_positions.items(), 5):
    print(
        f"\t timestamp {int(timestamp.total_seconds() * 1e9)} ns has {len(points)} observed points in slam-front-left view. "
    )
print("\t ...")


In [ ]:
import rerun as rr
import numpy as np
from projectaria_tools.utils.rerun_helpers import AriaGlassesOutline, ToTransform3D

print("=== Visualizing MPS trajectory and point cloud in 3D ===")

rr.init("MPS Trajectory Visualization")
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)

# The full filtered cloud is static: it does not change over the sequence.
rr.log(
    "world/semidense_points",
    rr.Points3D(positions=points_positions, colors=[255, 255, 255, 125], radii=0.001),
    static=True,
)

# Aria glass outline for visualization purpose
device_calib = vrs_data_provider.get_device_calibration()
aria_glasses_point_outline = AriaGlassesOutline(device_calib, use_cad_calib=True)

closed_loop_traj_cached_full = []
observation_points_cached = None
observation_colors_cached = None

for closed_loop_pose in trajectory_segment:
    capture_timestamp_ns = int(closed_loop_pose.tracking_timestamp.total_seconds() * 1e9)
    rr.set_time("device_time", duration=np.timedelta64(capture_timestamp_ns, "ns"))

    T_world_device = closed_loop_pose.transform_world_device

    # Log device pose as a coordinate frame
    rr.log("world/device", ToTransform3D(T_world_device))
    rr.log("world/device", rr.TransformAxes3D(axis_length=0.05))

    # Plot Aria glass outline
    rr.log(
        "world/device/glasses_outline",
        rr.LineStrips3D(aria_glasses_point_outline, colors=[150, 200, 40], radii=5e-3),
    )

    # Plot gravity direction vector
    rr.log(
        "world/gravity",
        rr.Arrows3D(
            origins=[T_world_device.translation()[0]],
            # length converted from 9.8 meter -> 10 cm
            vectors=[closed_loop_pose.gravity_world * 1e-2],
            colors=[101, 67, 33],
            radii=5e-3,
        ),
        static=False,
    )

    # Observations arrive far less often than the 1 kHz trajectory, so the last set is
    # held until the next one rather than blinking out between camera frames.
    if closed_loop_pose.tracking_timestamp in timestamp_to_point_positions:
        observation_points_cached = timestamp_to_point_positions[closed_loop_pose.tracking_timestamp]
        observation_colors_cached = timestamp_to_point_colors[closed_loop_pose.tracking_timestamp]
    if observation_points_cached is not None:
        rr.log(
            "world/semidense_observations",
            rr.Points3D(
                positions=observation_points_cached,
                colors=observation_colors_cached,
                radii=0.01,
            ),
            static=False,
        )

    # Plot the trajectory accumulated so far
    closed_loop_traj_cached_full.append(T_world_device.translation()[0])
    rr.log(
        "world/trajectory",
        rr.LineStrips3D(closed_loop_traj_cached_full, colors=[173, 216, 255], radii=5e-3),
        static=False,
    )

rr.notebook_show()

---

## Related tutorials

- `Tutorial_1_vrs_data_provider_basics` — the query APIs used for the on-device streams
- `Tutorial_2_device_calibration` — the `Device` frame these poses are expressed against
- `Tutorial_5_mps_basics` — MPS output layout, `MpsDataPathsProvider` and `MpsDataProvider`
- `Tutorial_7_hand_tracking` — the other algorithm with both an on-device and an MPS source